# NFL Game Flow Prediction Model

Predicts **how a game will unfold**, not just who covers: points by quarter, who scores first, halftime lead,
middle-eight points, who scores last before half and first after, total plays, total drives, and the final score.

Built on the hockey "game flow" idea: score state changes how teams play (leaders run and slow down, trailers
pass and speed up), so efficiency stats are also given in a **score-adjusted** version.

Data: nflverse play-by-play (public, downloads automatically). Only NFL stats.
Runtime in Colab: about 4-6 minutes for the full 2015-2026 load.

**How to use:** Runtime > Run all. Then edit the matchup list in Section 8 and re-run that cell.

**Injuries (optional):** upload `injury_load_hist.csv` from the project folder with the folder icon on the left
before running. Without it the model runs without injuries.

In [ ]:
import pandas as pd, numpy as np, warnings
from sklearn.linear_model import RidgeCV, LogisticRegressionCV, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

SEASONS = list(range(2015, 2027))
TEST_FROM = 2023      # model is fit on 2015-2022 and graded on 2023-2026
LOCAL = None          # folder with pbp_YYYY.csv.gz + games.csv, or None to download from nflverse
INJURY_FILE = "injury_load_hist.csv"   # optional: upload from the project folder (nfl-model/upgrades/data/)

COLS = ["game_id", "play_id", "season", "week", "season_type", "posteam", "defteam", "home_team", "away_team",
        "play_type", "pass", "rush", "qb_dropback", "qb_kneel", "qb_spike", "epa", "success", "wp",
        "score_differential", "qtr", "down", "ydstogo", "yardline_100", "yards_gained", "game_seconds_remaining",
        "half_seconds_remaining", "fixed_drive", "fixed_drive_result", "third_down_converted", "third_down_failed",
        "fourth_down_converted", "fourth_down_failed", "penalty", "penalty_team", "penalty_yards", "sack",
        "interception", "fumble_lost", "field_goal_result", "total_home_score", "total_away_score"]

def load(y):
    src = f"{LOCAL}/pbp_{y}.csv.gz" if LOCAL else \
        f"https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{y}.csv.gz"
    return pd.read_csv(src, usecols=COLS, low_memory=False)

raw = pd.concat([load(y) for y in SEASONS], ignore_index=True)
raw = raw[raw.season_type == "REG"].copy()
games = pd.read_csv(f"{LOCAL}/games.csv" if LOCAL else
                    "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv")
FIX = {"OAK": "LV", "SD": "LAC", "STL": "LA"}
for c in ["home_team", "away_team"]:
    games[c] = games[c].replace(FIX)
games = games[games.game_type == "REG"]
print(raw.shape, "plays")

## 1. Scoring timeline (needed for quarter points, middle eight, first/last score)

In [ ]:
raw = raw.sort_values(["game_id", "play_id"])
for side in ["home", "away"]:
    s = raw.groupby("game_id")[f"total_{side}_score"].transform(lambda x: x.ffill().fillna(0))
    raw[f"{side}_pts_play"] = s.groupby(raw.game_id).diff().fillna(s).clip(lower=0)
sc = raw[(raw.home_pts_play > 0) | (raw.away_pts_play > 0)].copy()
sc["scorer"] = np.where(sc.home_pts_play > 0, sc.home_team, sc.away_team)
sc["half"] = np.where(sc.qtr <= 2, 1, np.where(sc.qtr <= 4, 2, 3))
sc["mid8"] = ((sc.qtr == 2) & (sc.half_seconds_remaining <= 240)) | ((sc.qtr == 3) & (sc.half_seconds_remaining >= 1560))
sc["two_min"] = sc.qtr.isin([2, 4]) & (sc.half_seconds_remaining <= 120)

def pts_by(mask_col=None, qtr=None):
    d = sc if mask_col is None else sc[sc[mask_col]]
    if qtr is not None: d = d[d.qtr == qtr]
    return d.groupby("game_id")[["home_pts_play", "away_pts_play"]].sum()

GF = games[games.result.notna()][["game_id", "season", "week", "home_team", "away_team", "home_score", "away_score",
                                  "result", "total", "spread_line", "total_line", "roof"]].set_index("game_id")
for q in [1, 2, 3, 4]:
    GF = GF.join(pts_by(qtr=q).rename(columns={"home_pts_play": f"h_q{q}", "away_pts_play": f"a_q{q}"}))
GF = GF.join(pts_by("mid8").rename(columns={"home_pts_play": "h_mid8", "away_pts_play": "a_mid8"}))
GF = GF.join(pts_by("two_min").rename(columns={"home_pts_play": "h_2min", "away_pts_play": "a_2min"}))
GF = GF.fillna({c: 0 for c in GF.columns if c[:2] in ("h_", "a_")})
first = sc.groupby("game_id").first()
last1 = sc[sc.half == 1].groupby("game_id").last()
first2 = sc[sc.half == 2].groupby("game_id").first()
GF["home_scores_first"] = (first.scorer == first.home_team).astype(float).reindex(GF.index)
GF["home_last_1h"] = (last1.scorer == last1.home_team).astype(float).reindex(GF.index)
GF["home_first_2h"] = (first2.scorer == first2.home_team).astype(float).reindex(GF.index)
GF["h_half"] = GF.h_q1 + GF.h_q2; GF["a_half"] = GF.a_q1 + GF.a_q2
GF["half_margin"] = GF.h_half - GF.a_half
GF["home_leads_half"] = (GF.half_margin > 0).astype(float)
GF["home_win"] = (GF.result > 0).astype(float)
GF["mid8_total"] = GF.h_mid8 + GF.a_mid8

## 2. Team-game stats (offense = "for", defense = "against")
Every stat in your list, plus the additions. One row per team per game.

In [ ]:
sp = raw[(raw["pass"].eq(1) | raw["rush"].eq(1)) & raw.qb_kneel.ne(1) & raw.qb_spike.ne(1) & raw.epa.notna()].copy()
# score-state adjustment (hockey score effects): EPA minus league average in the same score bucket x quarter
sp["state"] = pd.cut(sp.score_differential, [-99, -17, -9, -4, -1, 0, 3, 8, 16, 99]).astype(str) + sp.qtr.clip(upper=4).astype(str)
sp["epa_adj"] = sp.epa - sp.groupby(["season", "state"]).epa.transform("mean")
sp["explosive"] = ((sp["pass"] == 1) & (sp.yards_gained >= 20)) | ((sp["rush"] == 1) & (sp.yards_gained >= 10))
sp["early"] = sp.down.isin([1, 2])
sp["neutral"] = (sp.qtr <= 3) & (sp.score_differential.abs() <= 7)
sp["lead7"] = sp.score_differential >= 7
sp["trail7"] = sp.score_differential <= -7
sp["h2"] = sp.qtr >= 3
sp["giveaway"] = sp.interception.fillna(0) + sp.fumble_lost.fillna(0)

def agg_side(d, key):
    g = d.groupby(["game_id", key])
    out = pd.DataFrame({
        "plays": g.size(),
        "epa": g.epa.mean(),
        "epa_adj": g.epa_adj.mean(),
        "early_sr": d[d.early].groupby(["game_id", key]).success.mean(),
        "explosive": g.explosive.mean(),
        "sack_rate": d[d.qb_dropback == 1].groupby(["game_id", key]).sack.mean(),
        "giveaways": g.giveaway.sum(),
        "neutral_pass": d[d.neutral].groupby(["game_id", key])["pass"].mean(),
        "pass_lead": d[d.lead7].groupby(["game_id", key])["pass"].mean(),
        "pass_trail": d[d.trail7].groupby(["game_id", key])["pass"].mean(),
        "epa_2h_minus_1h": d[d.h2].groupby(["game_id", key]).epa.mean() - d[~d.h2].groupby(["game_id", key]).epa.mean(),
    })
    out.index.names = ["game_id", "team"]
    return out

# 3rd and 4th down
dn = raw[raw.down.isin([3, 4]) & raw.posteam.notna()].copy()
d3 = dn[(dn.down == 3) & (dn.third_down_converted.fillna(0) + dn.third_down_failed.fillna(0) > 0)]
d4 = dn[(dn.down == 4) & dn.play_type.isin(["pass", "run", "punt", "field_goal"])]
def third(d, key):
    g = d.groupby(["game_id", key])
    o = pd.DataFrame({"third_conv": g.third_down_converted.mean(), "third_ytg": g.ydstogo.mean(), "third_att": g.size()})
    o.index.names = ["game_id", "team"]; return o
def fourth(d, key):
    o = d.assign(go=d.play_type.isin(["pass", "run"])).groupby(["game_id", key]).go.mean().to_frame("fourth_go")
    o.index.names = ["game_id", "team"]; return o

# drives: red zone trips, RZ TD%, points/drive, three-and-out, start field position
dr = (sp.groupby(["game_id", "posteam", "fixed_drive"])
        .agg(defteam=("defteam", "first"), result=("fixed_drive_result", "first"), min_yl=("yardline_100", "min"),
             start_yl=("yardline_100", "first"), n=("play_id", "size")).reset_index())
dr["pts"] = dr.result.map({"Touchdown": 7, "Field goal": 3}).fillna(0)
dr["rz"] = dr.min_yl <= 20
dr["rz_td"] = dr.rz & (dr.result == "Touchdown")
dr["three_out"] = (dr.n <= 3) & (dr.result == "Punt")
dr["own_start"] = 100 - dr.start_yl
def drives(d, key):
    g = d.groupby(["game_id", key])
    o = pd.DataFrame({"drives": g.size(), "pts_drive": g.pts.mean(), "rz_trips": g.rz.sum(),
                      "rz_td_pct": d[d.rz].groupby(["game_id", key]).rz_td.mean(), "three_out": g.three_out.mean(),
                      "start_fp": g.own_start.mean()})
    o.index.names = ["game_id", "team"]; return o

# pace: seconds per snap in neutral situations
pc = sp[sp.neutral].sort_values(["game_id", "play_id"]).copy()
pc["gap"] = -pc.groupby(["game_id", "fixed_drive", "qtr"]).game_seconds_remaining.diff()
pace = pc[(pc.gap > 0) & (pc.gap <= 60)].groupby(["game_id", "posteam"]).gap.mean().to_frame("pace")
pace.index.names = ["game_id", "team"]

# penalties (all plays) and special teams
pen = raw[raw.penalty == 1].groupby(["game_id", "penalty_team"]).agg(penalties=("penalty", "size"), pen_yds=("penalty_yards", "sum"))
pen.index.names = ["game_id", "team"]
st = raw[raw.play_type.isin(["punt", "kickoff", "field_goal", "extra_point"]) & raw.epa.notna()]
st_for = st.groupby(["game_id", "posteam"]).epa.sum(); st_ag = st.groupby(["game_id", "defteam"]).epa.sum()
st_for.index.names = st_ag.index.names = ["game_id", "team"]
stt = st_for.sub(st_ag, fill_value=0).to_frame("st_epa")
fg = raw[raw.play_type == "field_goal"].assign(made=lambda x: x.field_goal_result.eq("made"))
fgp = fg.groupby(["game_id", "posteam"]).made.mean().to_frame("fg_pct"); fgp.index.names = ["game_id", "team"]

OFF = agg_side(sp, "posteam").join(third(d3, "posteam")).join(fourth(d4, "posteam")).join(drives(dr, "posteam")) \
        .join(pace).join(pen).join(stt).join(fgp)
DEF = agg_side(sp, "defteam").join(third(d3, "defteam")).join(drives(dr, "defteam"))
DEF = DEF[["plays", "epa", "epa_adj", "early_sr", "explosive", "sack_rate", "giveaways", "third_conv", "third_ytg",
           "drives", "pts_drive", "rz_trips", "rz_td_pct", "three_out", "start_fp"]]
TG = OFF.add_prefix("o_").join(DEF.add_prefix("d_")).reset_index()
TG["team"] = TG.team.replace(FIX)
TG[["o_penalties", "o_pen_yds"]] = TG[["o_penalties", "o_pen_yds"]].fillna(0)

# flow outcomes from the team's view, so they can be averaged as "tendencies" too
long = pd.concat([
    GF.reset_index().assign(team=lambda x: x.home_team, opp=lambda x: x.away_team, pf=lambda x: x.home_score,
        pa=lambda x: x.away_score, mid8_f=lambda x: x.h_mid8, mid8_a=lambda x: x.a_mid8,
        twomin_f=lambda x: x.h_2min, first_score=lambda x: x.home_scores_first, last1h=lambda x: x.home_last_1h,
        first2h=lambda x: x.home_first_2h, half_m=lambda x: x.half_margin, home=1,
        **{f"q{q}_f": (lambda q: lambda x: x[f"h_q{q}"])(q) for q in range(1, 5)},
        **{f"q{q}_a": (lambda q: lambda x: x[f"a_q{q}"])(q) for q in range(1, 5)}),
    GF.reset_index().assign(team=lambda x: x.away_team, opp=lambda x: x.home_team, pf=lambda x: x.away_score,
        pa=lambda x: x.home_score, mid8_f=lambda x: x.a_mid8, mid8_a=lambda x: x.h_mid8,
        twomin_f=lambda x: x.a_2min, first_score=lambda x: 1 - x.home_scores_first, last1h=lambda x: 1 - x.home_last_1h,
        first2h=lambda x: 1 - x.home_first_2h, half_m=lambda x: -x.half_margin, home=0,
        **{f"q{q}_f": (lambda q: lambda x: x[f"a_q{q}"])(q) for q in range(1, 5)},
        **{f"q{q}_a": (lambda q: lambda x: x[f"h_q{q}"])(q) for q in range(1, 5)})])
long["double_dip"] = long.last1h * long.first2h      # scored last in 1H AND first in 2H
long["blown_lead"] = ((long.half_m > 0) & (long.pf < long.pa)).astype(float)
long["comeback"] = ((long.half_m < 0) & (long.pf > long.pa)).astype(float)
TG = TG.merge(long[["game_id", "team", "season", "week", "home", "pf", "pa", "mid8_f", "mid8_a", "twomin_f",
                    "first_score", "last1h", "first2h", "double_dip", "blown_lead", "comeback"] +
                   [f"q{q}_{s}" for q in range(1, 5) for s in "fa"]], on=["game_id", "team"])

# injuries (pregame, snap-weighted starters Out/Doubtful; from the project's injury layer)
try:
    J = pd.read_csv(INJURY_FILE).rename(columns={"off": "inj_off", "def": "inj_def"})
    J["team"] = J.team.replace(FIX)
    TG = TG.merge(J, on=["team", "season", "week"], how="left").fillna({"inj_off": 0, "inj_def": 0})
    HAVE_INJ = True
except FileNotFoundError:
    print("No injury file found; running without injuries."); HAVE_INJ = False
print(TG.shape, "team-games")

## 3. Pregame ratings: season-to-date averages, shrunk toward last season early in the year
rating = (sum of this season's prior games + 3 x last season's average) / (games played + 3).
Only games before the one being predicted are used.

In [ ]:
STATS = [c for c in TG.columns if c.startswith(("o_", "d_"))] + \
        ["pf", "pa", "mid8_f", "mid8_a", "twomin_f", "first_score", "last1h", "first2h", "double_dip",
         "blown_lead", "comeback"] + [f"q{q}_{s}" for q in range(1, 5) for s in "fa"]
K = 3
TG = TG.sort_values(["team", "season", "week"]).reset_index(drop=True)
season_avg = TG.groupby(["team", "season"])[STATS].mean()
prev = season_avg.copy(); prev.index = prev.index.set_levels(prev.index.levels[1] + 1, level=1)
lg = TG.groupby("season")[STATS].mean()
grp = TG.groupby(["team", "season"])
sh = grp[STATS].shift()
cum = sh.fillna(0).groupby([TG.team, TG.season]).cumsum()
cnt = sh.notna().astype(int).groupby([TG.team, TG.season]).cumsum()
pri = prev.reindex(pd.MultiIndex.from_frame(TG[["team", "season"]])).reset_index(drop=True)
pri = pri.fillna(lg.reindex(TG.season).reset_index(drop=True))
R = (cum + K * pri) / (cnt + K)
R.columns = ["r_" + c for c in STATS]
TG = pd.concat([TG, R], axis=1)
TG["games_played"] = grp.cumcount()
RCOLS = list(R.columns)
if HAVE_INJ: RCOLS += ["inj_off", "inj_def"]   # injuries are known pregame, used as-is for that week

## 4. Build the matchup table (home ratings vs away ratings)

In [ ]:
h = TG[["game_id", "team"] + RCOLS + ["games_played"]].add_prefix("H_").rename(columns={"H_game_id": "game_id", "H_team": "home_team"})
a = TG[["game_id", "team"] + RCOLS + ["games_played"]].add_prefix("A_").rename(columns={"A_game_id": "game_id", "A_team": "away_team"})
M = GF.reset_index().merge(h, on=["game_id", "home_team"]).merge(a, on=["game_id", "away_team"])
M = M[M.week >= 2].copy()
M["indoor"] = M.roof.isin(["dome", "closed"]).astype(float)
FEATS = [c for c in M.columns if c.startswith(("H_r_", "A_r_", "H_inj", "A_inj"))] + ["indoor"]
M = M.dropna(subset=FEATS)
train, test = M[M.season < TEST_FROM], M[M.season >= TEST_FROM]
print(len(train), "train games,", len(test), "test games,", len(FEATS), "features")

## 5. Fit one model per flow outcome and grade it on unseen seasons
Numeric outcomes use ridge regression; yes/no outcomes use logistic regression. Graded on 2023-2026, which the model
never saw. "Gain vs average" = how much the model beats just guessing the league average every game.

In [ ]:
NUM_T = ["h_q1", "a_q1", "h_q2", "a_q2", "h_q3", "a_q3", "h_q4", "a_q4", "h_half", "a_half", "half_margin",
         "h_mid8", "a_mid8", "mid8_total", "home_score", "away_score", "result", "total"]
BIN_T = ["home_scores_first", "home_leads_half", "home_last_1h", "home_first_2h", "home_win"]
plays = TG.groupby("game_id").o_plays.sum(); drv = TG.groupby("game_id").o_drives.sum()
for df in (M, train, test):
    df["total_plays"] = df.game_id.map(plays); df["total_drives"] = df.game_id.map(drv)
NUM_T += ["total_plays", "total_drives"]

MODELS, rows = {}, []
for t in NUM_T:
    tr, te = train.dropna(subset=[t]), test.dropna(subset=[t])
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(0, 4, 20))).fit(tr[FEATS], tr[t])
    p = m.predict(te[FEATS]); base = np.abs(te[t] - tr[t].mean()).mean(); mae = np.abs(te[t] - p).mean()
    MODELS[t] = m
    rows.append({"outcome": t, "type": "points/count", "test_MAE": round(mae, 2), "avg-guess_MAE": round(base, 2),
                 "gain_vs_avg_%": round(100 * (1 - mae / base), 1), "corr": round(np.corrcoef(p, te[t])[0, 1], 3)})
for t in BIN_T:
    tr, te = train.dropna(subset=[t]), test.dropna(subset=[t])
    m = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=10, max_iter=2000)).fit(tr[FEATS], tr[t])
    p = m.predict_proba(te[FEATS])[:, 1]; b = tr[t].mean()
    brier, base = ((p - te[t]) ** 2).mean(), ((b - te[t]) ** 2).mean()
    MODELS[t] = m
    rows.append({"outcome": t, "type": "yes/no", "test_MAE": round(brier, 3), "avg-guess_MAE": round(base, 3),
                 "gain_vs_avg_%": round(100 * (1 - brier / base), 1),
                 "corr": round(((p > .5) == (te[t] > .5)).mean(), 3)})
GRADE = pd.DataFrame(rows)
print("For yes/no outcomes the MAE columns are Brier scores and corr is hit rate at 50%.")
print(GRADE.to_string(index=False))

# market comparison where lines exist (context only)
mk = test.dropna(subset=["spread_line", "total_line"])
print("\nMarket for reference on the same test games: spread MAE",
      round(np.abs(mk.result - mk.spread_line).mean(), 2), "| model margin MAE",
      round(np.abs(mk.result - MODELS["result"].predict(mk[FEATS])).mean(), 2),
      "| total line MAE", round(np.abs(mk.total - mk.total_line).mean(), 2), "| model total MAE",
      round(np.abs(mk.total - MODELS["total"].predict(mk[FEATS])).mean(), 2))

## 6. Which stats actually help? (drop-one-stat test)
For each stat, refit every model without it (both teams' for/against versions) and measure how much worse the
test-season predictions get. Bigger number = that stat carries information the others do not.

In [ ]:
import re
def base_name(c):
    return re.sub(r"^(o_|d_)", "", re.sub(r"^(H_r_|A_r_|H_|A_)", "", c))
GROUPS = {}
for c in FEATS: GROUPS.setdefault(base_name(c), []).append(c)
KEY_T = ["total", "result", "half_margin", "mid8_total", "total_plays", "home_scores_first"]
imp = {}
for gname, cols in GROUPS.items():
    keep = [c for c in FEATS if c not in cols]; row = {}
    for t in KEY_T:
        tr, te = train.dropna(subset=[t]), test.dropna(subset=[t])
        if t in BIN_T:
            m = make_pipeline(StandardScaler(), LogisticRegression(C=MODELS[t][-1].C_[0], max_iter=2000)).fit(tr[keep], tr[t])
            full = ((MODELS[t].predict_proba(te[FEATS])[:, 1] - te[t]) ** 2).mean()
            row[t] = ((m.predict_proba(te[keep])[:, 1] - te[t]) ** 2).mean() / full - 1
        else:
            m = make_pipeline(StandardScaler(), Ridge(alpha=MODELS[t][-1].alpha_)).fit(tr[keep], tr[t])
            full = ((MODELS[t].predict(te[FEATS]) - te[t]) ** 2).mean()
            row[t] = ((m.predict(te[keep]) - te[t]) ** 2).mean() / full - 1
    imp[gname] = row
IMP = (pd.DataFrame(imp).T * 100).round(2)
IMP["avg"] = IMP.mean(axis=1).round(2)
print("% increase in test error when the stat is removed (negative = model is better without it)")
print(IMP.sort_values("avg", ascending=False).to_string())

## 7. Team flow profiles (current season)
Your stat list per team, averaged over this season's games.

In [ ]:
cur = TG[TG.season == TG.season.max()]
show = {"o_third_conv": "3rd% off", "d_third_conv": "3rd% def", "o_third_ytg": "3rd ytg off", "d_third_ytg": "3rd ytg def",
        "o_rz_trips": "RZ trips", "d_rz_trips": "RZ trips allowed", "o_rz_td_pct": "RZ TD% off", "d_rz_td_pct": "RZ TD% def",
        "mid8_f": "mid8 pts for", "mid8_a": "mid8 pts ag", "double_dip": "double dip %", "o_penalties": "penalties",
        "o_pen_yds": "pen yds", "o_pace": "pace (s/snap)", "o_plays": "off plays", "d_plays": "def plays",
        "o_epa_adj": "adj EPA off", "d_epa_adj": "adj EPA def", "o_explosive": "explosive off", "o_three_out": "3&out off",
        "first_score": "scores first %"}
prof = cur.groupby("team")[list(show)].mean().rename(columns=show)
prof["total plays/g"] = prof["off plays"] + prof["def plays"]
if HAVE_INJ: prof["inj load (latest)"] = cur.sort_values("week").groupby("team")[["inj_off", "inj_def"]].last().sum(axis=1)
print(prof.round(3).sort_values("adj EPA off", ascending=False).to_string())

## 8. Predict the flow of upcoming games
Uses each team's latest rating (all games so far). Edit `MATCHUPS` or leave it None for the next unplayed week.

In [ ]:
MATCHUPS = None   # e.g. [("BUF", "MIA"), ("CHI", "NO")]  as (home, away)
INJ_OVERRIDE = {}  # e.g. {"BUF": (1.5, 0.8)}  this week's (offense, defense) injury load, if known

last_rating = {}
for team, d in TG[TG.season == TG.season.max()].groupby("team"):
    d = d.sort_values("week")
    n = len(d); s = d[STATS].sum(); p = prev.loc[(team, TG.season.max())] if (team, TG.season.max()) in prev.index else lg.iloc[-1]
    last_rating[team] = ((s + K * p) / (n + K)).add_prefix("r_")

def matchup_row(home, away, indoor=0.0):
    r = {}
    for side, tm in (("H_", home), ("A_", away)):
        for k, v in last_rating[tm].items(): r[side + k] = v
        io, idf = INJ_OVERRIDE.get(tm, (0.0, 0.0))
        r[side + "inj_off"], r[side + "inj_def"] = io, idf
    r["indoor"] = indoor
    return pd.DataFrame([r])[FEATS]

if MATCHUPS is None:
    up = games[(games.season == TG.season.max()) & games.result.isna()]
    up = up[up.week == up.week.min()]
    MATCHUPS = list(zip(up.home_team, up.away_team, up.roof.isin(["dome", "closed"]).astype(float)))
out = []
for mu in MATCHUPS:
    home, away = mu[0], mu[1]; indoor = mu[2] if len(mu) > 2 else 0.0
    x = matchup_row(home, away, indoor)
    p = {t: MODELS[t].predict(x)[0] for t in NUM_T}; p.update({t: MODELS[t].predict_proba(x)[0, 1] for t in BIN_T})
    out.append({"game": f"{away} @ {home}",
                "Q1": f"{p['a_q1']:.1f}-{p['h_q1']:.1f}", "Q2": f"{p['a_q2']:.1f}-{p['h_q2']:.1f}",
                "Q3": f"{p['a_q3']:.1f}-{p['h_q3']:.1f}", "Q4": f"{p['a_q4']:.1f}-{p['h_q4']:.1f}",
                "half": f"{p['a_half']:.1f}-{p['h_half']:.1f}", "final": f"{p['away_score']:.1f}-{p['home_score']:.1f}",
                "total": round(p["total"], 1), "home margin": round(p["result"], 1),
                "mid8 pts": round(p["mid8_total"], 1), "plays": round(p["total_plays"]), "drives": round(p["total_drives"], 1),
                f"home scores 1st": f"{p['home_scores_first']:.0%}", "home leads at half": f"{p['home_leads_half']:.0%}",
                "home last 1H": f"{p['home_last_1h']:.0%}", "home first 2H": f"{p['home_first_2h']:.0%}",
                "home win": f"{p['home_win']:.0%}"})
PRED = pd.DataFrame(out)
print("Scores shown away-home.")
print(PRED.to_string(index=False))

## 9. Game flow chart for a finished game
Cumulative score-adjusted EPA margin (home minus away) with home win probability. Edit GAME.

In [ ]:
import matplotlib.pyplot as plt
def flow_chart(game_id):
    x = sp[sp.game_id == game_id].sort_values("play_id")
    if x.empty: print("no plays for", game_id); return
    t = (3600 - x.game_seconds_remaining) / 60
    flow = np.where(x.posteam == x.home_team, x.epa_adj, -x.epa_adj).cumsum()
    wp = np.where(x.posteam == x.home_team, x.wp, 1 - x.wp)
    fig, ax = plt.subplots(figsize=(10, 4)); ax.plot(t, flow, lw=2, color="#1f4e79", label="flow (home - away)")
    ax.axhline(0, color="grey", lw=.8); [ax.axvline(q, color="lightgrey", ls="--", lw=.8) for q in (15, 30, 45)]
    ax2 = ax.twinx(); ax2.plot(t, wp, color="#c55a11", alpha=.7, lw=1, label="home win prob"); ax2.set_ylim(0, 1)
    ax.set_title(f"{x.away_team.iloc[0]} @ {x.home_team.iloc[0]}   ({game_id})"); ax.set_xlabel("game minute")
    fig.legend(loc="upper left", bbox_to_anchor=(.08, .88), fontsize=8); plt.tight_layout(); plt.show()

GAME = GF.index[-1]
flow_chart(GAME)